# F1 + Clima — Narrativa de Ingeniería de Datos

> **Autor:** Manuel cartin
> **Enfoque** Data Engineer
> **Período:** Temporadas F1 2019–2024  
> **Objetivo:** Documentar el pipeline de integración de datos que une resultados de carrera con condiciones meteorológicas, incluyendo los retos técnicos encontrados en el camino.

---

Este notebook **no es un análisis de predicción climática**. Es la bitácora técnica del proceso de ingeniería de datos necesario para construir el dataset que haría posible ese análisis. Cada sección documenta una decisión de diseño real, un problema encontrado y la solución aplicada.

### ¿Por qué este pipeline es no trivial?

Tres fuentes de datos completamente heterogéneas que no comparten ninguna clave directa:

| Fuente | Clave disponible | Lo que necesitamos |
|--------|-----------------|--------------------|
| Resultados F1 (CSV) | Nombre de circuito (`Track`) | Fecha + clima |
| Calendario F1 (XLSX) | Nombre de circuito + fecha | Puente entre fuentes |
| Datos meteorológicos (CSV por ciudad) | Ciudad + fecha diaria | Circuito al que pertenece |

**El problema central:** los CSVs de F1 no tienen fecha, y los CSVs de clima no saben que existen circuitos de F1. Construir el puente entre ambos mundos es el trabajo real de este pipeline.

## 1. Fuentes de Datos y Decisiones de Proximidad Geográfica

Los datos meteorológicos provienen de estaciones de ciudades cercanas a los circuitos, **no de los circuitos mismos**. Esta es una limitación conocida y documentada de esta PoC. La siguiente tabla registra la decisión de proximidad tomada para cada Gran Premio:

| Gran Premio | Ciudad del Circuito | Ciudad en CSV Meteo | Distancia Aprox. |
|---|---|---|---|
| GP de Emilia Romagna | Imola, Italia | Forlì (`export-forl-0.csv`) | ~40 km |
| GP de Arabia Saudita | Jeddah | Jeddah (`export-djedda0.csv`) | ~10 km |
| GP de Abu Dhabi | Abu Dhabi | Abu Dhabi (`export-abu-dhabi0.csv`) | ~10 km |
| GP de China | Shanghái | Shanghái (`export-shanghai0.csv`) | ~30 km |
| GP de Las Vegas | Las Vegas, NV | Las Vegas (`export-las-vegas0.csv`) | ~5 km |
| GP de Brasil | São Paulo | São Paulo (`export-s-o-paulo0.csv`) | ~20 km |
| GP de México | Ciudad de México | Ciudad de México (`export-mexico0.csv`) | ~15 km |
| GP de EE. UU. (COTA) | Austin, TX | Austin (`export-austin0.csv`) | ~25 km |
| GP de Qatar | Lusail | Doha (`export-doha0.csv`) | ~20 km |
| GP de Japón | Suzuka | Nagoya (`export-nagoya0.csv`) | ~60 km |
| GP de Italia | Monza | Monza (`export-monza0.csv`) | ~5 km |
| GP de Países Bajos | Zandvoort | Zandvoort (`export-zandvoort0.csv`) | ~5 km |
| GP de Hungría | Mogyoród | Budapest (`export-budapest0.csv`) | ~20 km |
| GP de Gran Bretaña | Silverstone | Northampton (`export-northampton0.csv`) | ~20 km |
| GP de Austria | Spielberg | Knittelfeld (`export-knittelfeld0.csv`) | ~15 km |
| GP de Canadá | Montreal | Montreal (`export-montreal0.csv`) | ~10 km |
| GP de España | Barcelona | Barcelona (`export-barcelone0.csv`) | ~10 km |
| GP de Mónaco | Mónaco | Mónaco (`export-monaco0.csv`) | ~5 km |
| GP de Azerbaiyán | Bakú | Bakú (`export-bakou0.csv`) | ~5 km |
| GP de Australia | Melbourne | Melbourne (`export-melbourne0.csv`) | ~10 km |
| GP de Miami | Miami, FL | Miami (`export-miami0.csv`) | ~10 km |
| GP de Baréin | Sakhir | Manama (`export-manama0.csv`) | ~25 km |
| GP de Singapur | Singapur | Singapur (`export-singapour0.csv`) | ~5 km |
| GP de Bélgica | Spa-Francorchamps | Durbuy (`export-durbuy0.csv`) | ~30 km |

> **Nota de ingeniería:** La distancia más grande es Suzuka → Nagoya (~60 km). En circuitos de montaña como Austria, incluso una distancia pequeña puede implicar diferencias de altitud significativas. Esta limitación debe considerarse al interpretar cualquier resultado climático.

## 2. Dependencias

In [ ]:
import pandas as pd
import numpy as np
import glob
from datetime import datetime

print('Librerías cargadas correctamente')
print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')

## 3. Reto 1 — Los CSVs de F1 No Tienen Fecha

### Problema

Los datasets públicos de resultados F1 identifican cada carrera por el nombre del circuito (`Track = 'Australia'`) pero **no incluyen la fecha del Gran Premio**. Sin fecha no hay forma de hacer join con datos meteorológicos, que son series temporales diarias.

```
df_f1_season_2019:
  Track | Position | Driver | Team | Starting Grid | ...
  -------------------------------------------------------
  Australia | 1 | Bottas | Mercedes | 2 | ...
  Australia | 2 | Hamilton | Mercedes | 1 | ...
  ❌ Sin columna Date
```

### Solución: Calendario como tabla de mapeo `Track → Date`

Se construyó manualmente un archivo XLSX por temporada que actúa como *lookup table*. El join se hace sobre la columna `Track` que sí comparten ambas fuentes.

> **Decisión de diseño — `how='left'`:** Se usa `left` y no `inner` deliberadamente. Un `NaN` en la columna `Date` post-merge es una **señal de alerta visible**: indica un circuito sin mapeo. Un `inner` join simplemente haría desaparecer esas filas sin ninguna advertencia.

In [ ]:
# --- TEMPORADA 2019 ---
df_f1_season_2019 = pd.read_csv('/content/sample_data/formula1_2019season_raceResults.csv')
print('Shape resultados 2019:', df_f1_season_2019.shape)
print('Columnas:', df_f1_season_2019.columns.tolist())
display(df_f1_season_2019.head())

In [ ]:
# El calendario es la tabla puente que aporta la dimensión temporal
df_calendar_2019 = pd.read_excel('/content/sample_data/2019_calendar.xlsx')
print('Columnas calendario:', df_calendar_2019.columns.tolist())
display(df_calendar_2019.head())

In [ ]:
df_f1_con_fechas_2019 = pd.merge(
    df_f1_season_2019,   # resultados — tiene Track, NO tiene Date
    df_calendar_2019,    # calendario — tiene Track + Date
    on='Track',
    how='left'           # left: NaN visible si un circuito no tiene fecha mapeada
)

# Diagnóstico post-merge: cuántos circuitos quedaron sin fecha
sin_fecha = df_f1_con_fechas_2019['Date'].isna().sum()
print(f'Filas sin fecha (circuitos no mapeados): {sin_fecha}')
display(df_f1_con_fechas_2019.head())

In [ ]:
# --- TEMPORADA 2020 ---
df_f1_season_2020 = pd.read_csv('/content/sample_data/formula1_2020season_raceResults.csv')
df_calendar_2020  = pd.read_excel('/content/sample_data/2020_calendar.xlsx')

df_f1_con_fechas_2020 = pd.merge(
    df_f1_season_2020, df_calendar_2020, on='Track', how='left'
)
print('Sin fecha 2020:', df_f1_con_fechas_2020['Date'].isna().sum())
display(df_f1_con_fechas_2020.head())

In [ ]:
# --- TEMPORADA 2021 ---
df_f1_season_2021 = pd.read_csv('/content/sample_data/formula1_2021season_raceResults.csv')
df_calendar_2021  = pd.read_excel('/content/sample_data/2021_calendar.xlsx')

df_f1_con_fechas_2021 = pd.merge(
    df_f1_season_2021, df_calendar_2021, on='Track', how='left'
)
print('Sin fecha 2021:', df_f1_con_fechas_2021['Date'].isna().sum())
display(df_f1_con_fechas_2021.head())

In [ ]:
# --- TEMPORADA 2022 ---
df_f1_season_2022 = pd.read_csv('/content/sample_data/Formula1_2022season_raceResults.csv')
df_calendar_2022  = pd.read_excel('/content/sample_data/2022_calendar.xlsx')

df_f1_con_fechas_2022 = pd.merge(
    df_f1_season_2022, df_calendar_2022, on='Track', how='left'
)
print('Sin fecha 2022:', df_f1_con_fechas_2022['Date'].isna().sum())
display(df_f1_con_fechas_2022.head())

In [ ]:
# --- TEMPORADA 2023 ---
df_season_2023   = pd.read_csv('/content/sample_data/Formula1_2023season_raceResults.csv')
df_calendar_2023 = pd.read_excel('/content/sample_data/2023_calendar.xlsx')

df_f1_con_fechas_2023 = pd.merge(
    df_season_2023, df_calendar_2023, on='Track', how='left'
)
print('Sin fecha 2023:', df_f1_con_fechas_2023['Date'].isna().sum())
display(df_f1_con_fechas_2023.head())

In [ ]:
# --- TEMPORADA 2024 ---
df_season_2024   = pd.read_csv('/content/sample_data/Formula1_2024season_raceResults.csv')
df_calendar_2024 = pd.read_excel('/content/sample_data/2024_calendar.xlsx')

df_f1_con_fechas_2024 = pd.merge(
    df_season_2024, df_calendar_2024, on='Track', how='left'
)
print('Sin fecha 2024:', df_f1_con_fechas_2024['Date'].isna().sum())
display(df_f1_con_fechas_2024.head())

In [ ]:
# Exportar archivos intermedios — buena práctica: auditar cada etapa por separado
df_f1_con_fechas_2019.to_csv('df_f1_con_fechas_2019.csv', index=False)
df_f1_con_fechas_2020.to_csv('df_f1_con_fechas_2020.csv', index=False)
df_f1_con_fechas_2021.to_csv('df_f1_con_fechas_2021.csv', index=False)
df_f1_con_fechas_2022.to_csv('df_f1_con_fechas_2022.csv', index=False)
df_f1_con_fechas_2023.to_csv('df_f1_con_fechas_2023.csv', index=False)
df_f1_con_fechas_2024.to_csv('df_f1_con_fechas_2024.csv', index=False)
print('Archivos intermedios exportados.')

## 4. Reto 2 — Consolidación Multi-Temporada: Schema Drift y Trazabilidad

### Problema

Concatenar 6 temporadas tiene dos trampas:

1. **Schema drift:** Kaggle publicó los CSVs en distintos momentos; los nombres de columnas no son idénticos entre años (mayúsculas inconsistentes, columnas extra).
2. **Pérdida de trazabilidad:** una vez concatenados, ya no se sabe de qué temporada viene cada fila si no se agrega una columna de origen.

### Solución: columna `season` + `.copy().reset_index()`

Se agrega la columna `season` *antes* de concatenar. El `.copy()` explícito post-concat garantiza independencia de memoria y evita el `SettingWithCopyWarning`. El `reset_index(drop=True)` produce un índice limpio y contiguo, necesario para que los merges posteriores sean predecibles.

In [ ]:
# Agregar columna de temporada ANTES de concatenar — trazabilidad de origen
df_season_2019_column = df_f1_con_fechas_2019.copy()
df_season_2019_column['season'] = 2019

df_season_2020_column = df_f1_con_fechas_2020.copy()
df_season_2020_column['season'] = 2020

df_season_2021_column = df_f1_con_fechas_2021.copy()
df_season_2021_column['season'] = 2021

df_season_2022_column = df_f1_con_fechas_2022.copy()
df_season_2022_column['season'] = 2022

df_season_2023_column = df_f1_con_fechas_2023.copy()
df_season_2023_column['season'] = 2023

df_season_2024_column = df_f1_con_fechas_2024.copy()
df_season_2024_column['season'] = 2024

print('Columna season agregada a las 6 temporadas.')

In [ ]:
lista_de_datasets = [
    df_season_2019_column,
    df_season_2020_column,
    df_season_2021_column,
    df_season_2022_column,
    df_season_2023_column,
    df_season_2024_column
]

# .copy() explícito: independencia de memoria antes de cualquier mutación
# reset_index(drop=True): índice limpio y contiguo — evita índices duplicados
#   que rompen merges y operaciones loc[] posteriores
df_unido = pd.concat(lista_de_datasets).copy().reset_index(drop=True)

print(f'Shape dataset F1 unificado: {df_unido.shape}')
print(f'Temporadas presentes: {sorted(df_unido["season"].unique())}')
print(f'Circuitos únicos: {df_unido["Track"].nunique()}')
display(df_unido.head())

In [ ]:
# Verificar tipos de datos — especialmente Date, que debe ser datetime antes del merge final
print(df_unido.dtypes)

# Alerta temprana: si Date es object aquí, fallará el merge climático
if df_unido['Date'].dtype == object:
    print('\n⚠️  Date es string (object). Se convertirá antes del merge climático.')
else:
    print('\n✓ Date ya es datetime.')

In [ ]:
df_unido.to_csv('df_season_2019_2024.csv', index=False)
print('df_season_2019_2024.csv exportado.')

## 5. Reto 3 — Los CSV Meteorológicos Tienen 3 Filas de Metadatos

### Problema

Los archivos exportados del servicio meteorológico incluyen 3 filas de cabecera antes de los datos reales (nombre del sitio, coordenadas, unidades). Leerlos sin configuración produce un DataFrame con nombres de columna incorrectos y todos los valores numéricos como `object`.

```
export-melbourne0.csv (primeras 5 líneas reales):
  Fila 1: ## Melbourne Weather Data
  Fila 2: ## Source: ...
  Fila 3: ## Units: Celsius, mm, km/h
  Fila 4: DATE,MAX_TEMPERATURE_C,MIN_TEMPERATURE_C,...   ← cabecera real
  Fila 5: 2009-01-01,26,20,...                           ← primer dato
```

### Solución: `skiprows=3` + etiquetar origen inmediatamente

El patrón de agregar `Track` justo después de la carga actúa como **clave de trazabilidad geográfica**. Si el DataFrame viaja por el pipeline y se concatena con otros, siempre se sabe de qué circuito proviene.

In [ ]:
# skiprows=3 salta las 3 filas de metadatos del exportador
# Track se asigna inmediatamente — antes de cualquier transformación
df_weather_australia = pd.read_csv('/content/sample_data/export-melbourne0.csv', skiprows=3)
df_weather_australia['Track'] = 'Australia'

print('Shape:', df_weather_australia.shape)
print('Columnas (primeras 8):', df_weather_australia.columns[:8].tolist())
display(df_weather_australia.head(3))

In [ ]:
df_weather_bahrein = pd.read_csv('/content/sample_data/export-manama0.csv', skiprows=3)
df_weather_bahrein['Track'] = 'Bahrain'
display(df_weather_bahrein.head(3))

In [ ]:
df_weather_china = pd.read_csv('/content/sample_data/export-shanghai0.csv', skiprows=3)
df_weather_china['Track'] = 'China'
display(df_weather_china.head(3))

In [ ]:
# Circuitos de temporadas 2020-2021 que no estaban en el consolidado principal
df_weather_rusia = pd.read_csv('/content/sample_data/export-sotchi0.csv', skiprows=3)
df_weather_rusia['Track'] = 'Russia'

df_weather_portugal = pd.read_csv('/content/sample_data/export-portimao0.csv', skiprows=3)
df_weather_portugal['Track'] = 'Portugal'

df_weather_turquia = pd.read_csv('/content/sample_data/export-istanbul0.csv', skiprows=3)
df_weather_turquia['Track'] = 'Turkey'

df_weather_alemania = pd.read_csv('/content/sample_data/export-baden-baden0.csv', skiprows=3)
df_weather_alemania['Track'] = 'Germany'

# GP 70th Anniversary se corre en Silverstone — mismo CSV que Gran Bretaña
df_weather_70th = pd.read_csv('/content/sample_data/export-northampton0.csv', skiprows=3)
df_weather_70th['Track'] = '70th Anniversary'

print('CSVs de circuitos extra cargados.')

## 6. Reto 4 — Ciudad ≠ Circuito: El Problema de Mapeo de Claves

### Problema

El CSV meteorológico consolidado (`df_consolidado.csv`) usa nombres de ciudad como clave (`Ciudad_Base = 'manama'`), pero el dataset de F1 usa nombres de Gran Premio (`Track = 'Bahrain'`). **No hay ninguna columna compartida entre ambas fuentes.**

Además, los nombres en el CSV meteo vienen del exportador en formato libre (minúsculas, con acentos, en francés a veces: `'barcelone'`, `'singapour'`, `'forl-'`), mientras que F1 usa inglés estandarizado.

### Solución: tabla de mapeo + normalización `.str.lower().str.strip()`

Se construyó un archivo `clima ciudad.xlsx` con tres columnas:
- `Ciudad del Dataset` → nombre tal como aparece en el CSV meteo
- `Ciudad del Circuito` → nombre del circuito en F1
- `Gran Premio` → nombre completo del Gran Premio

La normalización de strings absorbe diferencias de case y espacios invisibles. Se usa `.map()` en lugar de `.merge()` porque es O(1) por lookup sobre un índice vs O(n) del merge, y porque el `NaN` resultante es explícito y auditable.

In [ ]:
# Dataset climático consolidado — contiene datos de todas las ciudades en formato largo
df_climate = pd.read_csv('/content/df_consolidado.csv')
print('Shape df_climate:', df_climate.shape)
print('Columnas:', df_climate.columns.tolist())
display(df_climate.head(3))

In [ ]:
# Tabla de mapeo: ciudad del dataset → nombre del circuito F1
# skiprows=2: este XLSX también tiene filas de título antes de los datos
df_city_weather = pd.read_excel('/content/clima ciudad.xlsx', skiprows=2)
df_map_city = df_city_weather.copy()
print('Columnas tabla de mapeo:', df_map_city.columns.tolist())
display(df_map_city.head())

In [ ]:
# PASO 1 — Normalizar claves en AMBOS lados antes de cualquier join
# .str.lower().str.strip() absorbe: mayúsculas, espacios iniciales/finales,
# diferencias de encoding que no se ven pero rompen la igualdad de strings
df_map_city['Ciudad_Clave_Limpia'] = df_map_city['Ciudad del Dataset'].str.lower().str.strip()
df_climate['Ciudad_Clave_Limpia']  = df_climate['Ciudad_Base'].str.lower().str.strip()

# PASO 2 — Construir la Serie de mapeo: Clave Limpia → Ciudad del Circuito
# set_index convierte la columna en índice hash — lookup O(1)
mapa_circuito = df_map_city.set_index('Ciudad_Clave_Limpia')['Ciudad del Circuito']

# PASO 3 — Aplicar el mapeo
# .map() devuelve NaN si la clave no existe en el mapa — auditable
df_climate['Gran_Premio'] = df_climate['Ciudad_Clave_Limpia'].map(mapa_circuito)

# Diagnóstico: ciudades sin mapeo a circuito F1
sin_mapeo = df_climate[df_climate['Gran_Premio'].isna()]['Ciudad_Base'].unique()
print(f'Ciudades sin mapeo a circuito F1: {len(sin_mapeo)}')
if len(sin_mapeo) > 0:
    print('  →', sin_mapeo)

# PASO 4 — Limpiar columnas auxiliares
df_climate = df_climate.drop('Ciudad_Clave_Limpia', axis=1)
df_map_city = df_map_city.drop('Ciudad_Clave_Limpia', axis=1)

display(df_climate.head(3))

In [ ]:
# Renombrar Gran_Premio a Track para homologar con el dataset de F1
df_climate = df_climate.rename(columns={'Gran_Premio': 'Track'})

df_climate.to_csv('df_climate.csv', index=False)
print('df_climate.csv exportado con columna Track.')

## 7. Reto 5 — El Merge Silencioso: Tipos de Fecha Inconsistentes

### Problema

Este es el bug más peligroso del pipeline porque **no produce ningún error**. Pandas ejecuta el merge, retorna un DataFrame con la forma correcta, y solo revisando los datos se descubre que todas las columnas climáticas son `NaN`.

**Causa:** `df_unido['Date']` es `object` (string con formato `'2019-03-17'`) y `df_climate['DATE']` también es `object` pero con formato diferente. Al comparar `'2019-03-17' == '2019-03-17'` debería funcionar, pero diferencias de encoding, espacios invisibles o zonas horarias rompen la igualdad silenciosamente.

**Regla de oro:** nunca hacer merge sobre columnas de fecha sin antes verificar `df.dtypes` y convertir ambas con `pd.to_datetime()` explícitamente.

### Síntoma diagnóstico

```python
# Post-merge: si este número es alto, el join falló
nans_clima = df_resultado['MAX_TEMPERATURE_C'].isna().sum()
total = len(df_resultado)
print(f'Filas sin clima: {nans_clima}/{total} ({100*nans_clima/total:.1f}%)')
```

In [ ]:
# Cargar el csv intermedio de clima (ya con Track) y los circuitos extra
df_climate_prep = pd.read_csv('/content/sample_data/df_climate.csv')

# Consolidar clima: circuitos principales + circuitos extra de 2020-2021
df_unido_climate = pd.concat([
    df_weather_70th,
    df_weather_rusia,
    df_weather_portugal,
    df_weather_turquia,
    df_weather_alemania,
    df_climate_prep
]).copy().reset_index(drop=True)

print(f'Shape clima consolidado: {df_unido_climate.shape}')
print(f'Circuitos en clima: {df_unido_climate["Track"].nunique()}')

In [ ]:
# PASO CRÍTICO — Homologar tipos de fecha antes del merge
# Sin esto el merge ejecuta sin error pero retorna 0 matches (NaN en todo el clima)
df_season_prep = pd.read_csv('/content/sample_data/df_season_2019_2024.csv')

# Homologar nombre de columna de fecha
df_season_prep = df_season_prep.rename(columns={'Date': 'DATE'})

# Convertir a datetime en AMBOS DataFrames
df_season_prep['DATE']   = pd.to_datetime(df_season_prep['DATE'],   errors='coerce')
df_unido_climate['DATE'] = pd.to_datetime(df_unido_climate['DATE'], errors='coerce')

print('Tipo DATE en df_season_prep:  ', df_season_prep['DATE'].dtype)
print('Tipo DATE en df_unido_climate:', df_unido_climate['DATE'].dtype)

# Fechas que no pudieron parsearse (serán NaN)
print(f'NaT en season: {df_season_prep["DATE"].isna().sum()}')
print(f'NaT en clima:  {df_unido_climate["DATE"].isna().sum()}')

In [ ]:
# MERGE FINAL: F1 + Clima, unidos por Track y fecha
df_all_circuit_2019_2024 = pd.merge(
    df_season_prep,
    df_unido_climate,
    on=['Track', 'DATE'],
    how='left'
)

# Diagnóstico inmediato post-merge
total = len(df_all_circuit_2019_2024)
nans_clima = df_all_circuit_2019_2024['MAX_TEMPERATURE_C'].isna().sum()
pct = 100 * nans_clima / total
print(f'Shape resultado: {df_all_circuit_2019_2024.shape}')
print(f'Filas sin dato climático: {nans_clima}/{total} ({pct:.1f}%)')

if pct > 20:
    print('⚠️  Más del 20% sin clima — revisar mapeo Track o tipos de fecha')
else:
    print('✓  Merge climático aceptable')

display(df_all_circuit_2019_2024.head())

## 8. Reto 6 — Duplicados por Unión Cruzada Temporal

### Problema

El dataset climático cubre múltiples años (desde 2009). Al hacer merge con los resultados F1, una combinación `Track + DATE` puede aparecer en el dataset de clima más de una vez si el CSV de esa ciudad tiene cobertura que se solapa con otro CSV cargado para el mismo circuito.

Los duplicados inflan artificialmente el dataset y cualquier estadística calculada sobre él.

### Patrón de diagnóstico: `keep=False` antes de eliminar

Siempre inspeccionar las filas duplicadas *antes* de eliminarlas para entender el origen del problema, no solo el síntoma.

In [ ]:
# Diagnóstico de duplicados
n_dup = df_all_circuit_2019_2024[['Track', 'DATE']].duplicated().sum()
print(f'Filas duplicadas (Track + DATE): {n_dup}')

if n_dup > 0:
    print('\nEjemplo de duplicados:')
    display(
        df_all_circuit_2019_2024[
            df_all_circuit_2019_2024.duplicated(subset=['Track', 'DATE'], keep=False)
        ].head(6)
    )

In [ ]:
# Eliminar duplicados — keep='first' preserva la fila más temprana del concat
df_all_circuit_2019_2024 = df_all_circuit_2019_2024.drop_duplicates(
    subset=['Track', 'DATE']
).reset_index(drop=True)

print(f'Shape post-deduplicación: {df_all_circuit_2019_2024.shape}')
print(f'Duplicados restantes: {df_all_circuit_2019_2024[["Track", "DATE"]].duplicated().sum()}')

## 9. Validación del Pipeline: Conteo de Columnas Esperadas vs Reales

Cuando se unen DataFrames con columnas solapadas (que no son las claves de join), Pandas agrega sufijos `_x` y `_y` automáticamente. Si este check falla, significa que hay columnas duplicadas en el resultado — un indicador de que algún merge no fue limpio.

In [ ]:
# Validación estructural del dataset final
claves_fusion = ['Track', 'DATE']

cols_left  = [c for c in df_season_prep.columns    if c not in claves_fusion]
cols_right = [c for c in df_unido_climate.columns  if c not in claves_fusion]

num_esperadas = len(claves_fusion) + len(cols_left) + len(cols_right)
num_reales    = df_all_circuit_2019_2024.shape[1]

print(f'Columnas esperadas : {num_esperadas}')
print(f'Columnas reales    : {num_reales}')

if num_reales == num_esperadas:
    print('✓  Estructura del merge correcta — sin columnas duplicadas')
else:
    cols_reales = set(df_all_circuit_2019_2024.columns)
    cols_esperadas_set = set(claves_fusion + cols_left + cols_right)
    extras   = cols_reales - cols_esperadas_set
    faltantes = cols_esperadas_set - cols_reales
    print('⚠️  Discrepancia detectada:')
    if extras:    print(f'  Columnas extra (revisar sufijos _x/_y): {extras}')
    if faltantes: print(f'  Columnas faltantes: {faltantes}')

In [ ]:
# Exportar dataset final
df_all_circuit_2019_2024.to_csv('df_all_circuit_2019_2024.csv', index=False)

print('='*50)
print('DATASET FINAL: df_all_circuit_2019_2024.csv')
print('='*50)
print(f'Filas    : {df_all_circuit_2019_2024.shape[0]}')
print(f'Columnas : {df_all_circuit_2019_2024.shape[1]}')
print(f'Circuitos: {df_all_circuit_2019_2024["Track"].nunique()}')
print(f'Temporadas: {sorted(df_all_circuit_2019_2024["season"].unique())}')
print('\nTipos de datos:')
print(df_all_circuit_2019_2024.dtypes)

## 10. Análisis Exploratorio sobre el Dataset Construido

Con el dataset ya integrado y limpio, se puede proceder al análisis. Las siguientes celdas son el punto de partida — el trabajo de ingeniería de los pasos anteriores es lo que hace que este código funcione.

### Variables clave

| Categoría | Variables | Rol |
|---|---|---|
| **Target** | `Position_Numeric`, `Points` | Variable a predecir/explicar |
| **Clima** | `PRECIP_TOTAL_DAY_MM`, `MAX_TEMPERATURE_C`, `WINDSPEED_MAX_KMH`, `WEATHER_CODE_*` | Features independientes principales |
| **Control** | `Starting Grid`, `Driver`, `Team`, `Track` | Para aislar el efecto del clima |
| **Contexto** | `Laps`, `Total Time/Gap`, `season` | Variables auxiliares |

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('/content/df_all_circuit_2019_2024.csv')
print('Filas y columnas:', df.shape)
df.head()

In [ ]:
# Limpieza de Position: convertir a numérico ('Ret', 'DSQ', 'NC' → NaN)
df['Position_Numeric'] = pd.to_numeric(df['Position'], errors='coerce')
df_clean = df.dropna(subset=['Position_Numeric', 'Starting Grid']).copy()

# Imputación de variables climáticas (media simple para análisis exploratorio)
climate_cols = ['PRECIP_TOTAL_DAY_MM', 'MAX_TEMPERATURE_C', 'WINDSPEED_MAX_KMH']
for col in climate_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mean())

print(f'Registros con posición numérica: {len(df_clean)}')
print(f'Descartados (retirados/DSQ/NC): {len(df) - len(df_clean)}')

In [ ]:
# Categorizar condición de carrera: Seca vs Mojada
PRECIP_THRESHOLD = 0.5  # mm — umbral para considerar carrera mojada

df_clean['Race_Condition'] = np.where(
    df_clean['PRECIP_TOTAL_DAY_MM'] > PRECIP_THRESHOLD,
    'Wet Race',
    'Dry Race'
)

print('Distribución de condición de carrera:')
print(df_clean['Race_Condition'].value_counts())

In [ ]:
# Impacto de la lluvia en la posición final
plt.figure(figsize=(10, 6))
sns.boxplot(
    x='Race_Condition', y='Position_Numeric', data=df_clean,
    palette={'Dry Race': 'steelblue', 'Wet Race': 'coral'}
)
plt.title('Distribución de Posición Final por Condición de Carrera', fontsize=14)
plt.xlabel('Condición de Carrera')
plt.ylabel('Posición Final')
plt.show()

In [ ]:
# Ganancia/pérdida de posiciones respecto a la salida
df_clean['Position_Gain_Loss'] = df_clean['Starting Grid'] - df_clean['Position_Numeric']

plt.figure(figsize=(10, 6))
sns.violinplot(
    x='Race_Condition', y='Position_Gain_Loss', data=df_clean,
    palette={'Dry Race': 'steelblue', 'Wet Race': 'coral'}
)
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.title('Ganancia/Pérdida de Posiciones: Seco vs Mojado', fontsize=14)
plt.xlabel('Condición de Carrera')
plt.ylabel('Posiciones ganadas (+ = mejoró)')
plt.show()

In [ ]:
# Relación entre posición de salida y posición final
plt.figure(figsize=(10, 6))
sns.regplot(
    x='Starting Grid', y='Position_Numeric', data=df_clean,
    scatter_kws={'alpha': 0.4, 'color': 'steelblue'},
    line_kws={'color': 'darkred'}
)
plt.title('Posición de Salida vs Posición Final', fontsize=14)
plt.xlabel('Starting Grid')
plt.ylabel('Position Final')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Efecto de la temperatura en el resultado
bins   = [0, 15, 25, df_clean['MAX_TEMPERATURE_C'].max() + 1]
labels = ['Frío (<15°C)', 'Templado (15-25°C)', 'Cálido (>25°C)']

df_clean['Temp_Range'] = pd.cut(df_clean['MAX_TEMPERATURE_C'], bins=bins, labels=labels, right=False)

plt.figure(figsize=(10, 6))
sns.boxplot(
    x='Temp_Range', y='Position_Numeric', data=df_clean,
    palette=['skyblue', 'lightgreen', 'salmon'], order=labels
)
plt.title('Posición Final por Rango de Temperatura Máxima', fontsize=14)
plt.xlabel('Rango de Temperatura')
plt.ylabel('Posición Final')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Mapa de correlación: variables de control + clima vs posición final
cols_heatmap = [
    'Starting Grid', 'Position_Numeric', 'Laps', 'Points',
    'MAX_TEMPERATURE_C', 'WINDSPEED_MAX_KMH', 'HUMIDITY_MAX_PERCENT',
    'PRESSURE_MAX_MB', 'CLOUDCOVER_AVG_PERCENT'
]

corr_matrix = df_clean[cols_heatmap].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=.5, cbar_kws={'label': 'Coeficiente de Correlación'}
)
plt.title('Mapa de Calor — Variables de Control y Climáticas vs Posición Final', fontsize=14)
plt.show()

In [ ]:
# Ranking de correlación con la posición final
corr_con_posicion = df_clean.select_dtypes(include=['float64', 'int64']).corr()['Position_Numeric']
print('Correlación con Position_Numeric (ordenada):')
print(corr_con_posicion.sort_values(ascending=False).to_string())

## 11. Modelado Predictivo (Random Forest)

Con el dataset limpio se construyen dos modelos para cuantificar el efecto del clima:
- **Modelo A (Baseline):** solo variables de control (`Starting Grid`, `Driver`, `Team`, `Track`)
- **Modelo B (Clima):** Modelo A + variables meteorológicas

Si el clima tiene poder predictivo real, el MAE del Modelo B debe ser menor que el del Modelo A.

In [ ]:
from sklearn.model_selection import KFold

TARGET = 'Position_Numeric'

def target_encode_kfold(df, col, target, n_splits=5):
    """Target encoding con K-Fold para evitar data leakage."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_means = np.zeros(len(df))

    for trn_idx, val_idx in kf.split(df):
        mapping = df.iloc[trn_idx].groupby(col)[target].mean()
        oof_means[val_idx] = df.iloc[val_idx][col].map(mapping)

    global_mean = df[target].mean()
    df[f'{col}_MeanEnc'] = df[col].map(df.groupby(col)[target].mean()).fillna(global_mean)
    return oof_means, df[f'{col}_MeanEnc']

df_clean['Driver_MeanEnc_OOF'], df_clean['Driver_MeanEnc'] = target_encode_kfold(df_clean, 'Driver', TARGET)
df_clean['Team_MeanEnc_OOF'],   df_clean['Team_MeanEnc']   = target_encode_kfold(df_clean, 'Team',   TARGET)

print('Target encoding completado para Driver y Team.')

In [ ]:
from sklearn.model_selection import train_test_split

# One-Hot Encoding para Track
df_final = pd.get_dummies(df_clean, columns=['Track'], prefix='Track', dtype=int)
track_cols = [c for c in df_final.columns if c.startswith('Track_')]

# Modelo A — Baseline (sin clima)
features_base = ['Starting Grid', 'Laps', 'Driver_MeanEnc_OOF', 'Team_MeanEnc_OOF'] + track_cols

# Modelo B — Con clima
features_clima = features_base + [
    'MAX_TEMPERATURE_C', 'WINDSPEED_MAX_KMH',
    'PRECIP_TOTAL_DAY_MM', 'HUMIDITY_MAX_PERCENT', 'CLOUDCOVER_AVG_PERCENT'
]

y = df_final[TARGET]

X_base  = df_final[features_base]
X_clima = df_final[features_clima]

X_base_train,  X_base_test,  y_train, y_test = train_test_split(X_base,  y, test_size=0.2, random_state=42)
X_clima_train, X_clima_test, _,       _      = train_test_split(X_clima, y, test_size=0.2, random_state=42)

print(f'Features Modelo A (baseline): {len(features_base)}')
print(f'Features Modelo B (clima):    {len(features_clima)}')
print(f'Entrenamiento: {len(X_base_train)} | Prueba: {len(X_base_test)}')

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# --- Modelo A: Baseline ---
model_base = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model_base.fit(X_base_train, y_train)
pred_base = model_base.predict(X_base_test)
mae_base = mean_absolute_error(y_test, pred_base)
r2_base  = r2_score(y_test, pred_base)

# --- Modelo B: Con Clima ---
model_clima = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model_clima.fit(X_clima_train, y_train)
pred_clima = model_clima.predict(X_clima_test)
mae_clima = mean_absolute_error(y_test, pred_clima)
r2_clima  = r2_score(y_test, pred_clima)

print('=' * 45)
print(f'{"":20} {"MAE":>8} {"R²":>8}')
print('-' * 45)
print(f'{"Modelo A (Baseline)":20} {mae_base:>8.3f} {r2_base:>8.4f}')
print(f'{"Modelo B (+ Clima)":20} {mae_clima:>8.3f} {r2_clima:>8.4f}')
print('=' * 45)

mejora = mae_base - mae_clima
print(f'\nMejora en MAE al agregar clima: {mejora:.3f} posiciones')
if mejora > 0:
    print('✓  El clima aporta poder predictivo')
else:
    print('— El clima no mejora el modelo en esta configuración')

In [ ]:
# Importancia de variables — Modelo B
feat_imp = pd.Series(model_clima.feature_importances_, index=features_clima)
top10 = feat_imp.sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
top10.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 10 Variables por Importancia — Modelo B (con Clima)', fontsize=13)
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

print('\nTop 10 features:')
print(top10.to_string())

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_clima_train, y_train)

best_model = grid_search.best_estimator_
pred_tuned = best_model.predict(X_clima_test)
mae_tuned  = mean_absolute_error(y_test, pred_tuned)

print(f'Mejores hiperparámetros: {grid_search.best_params_}')
print(f'MAE Modelo B optimizado: {mae_tuned:.3f}')
print(f'MAE Modelo B base      : {mae_clima:.3f}')
print(f'Ganancia por tuning    : {mae_clima - mae_tuned:.3f}')

## 12. Conclusiones de Ingeniería y Próximos Pasos

### Lo que este pipeline resolvió

| Reto | Solución aplicada | Resultado |
|---|---|---|
| CSV F1 sin fechas | Calendario XLSX como tabla puente + `merge left` | 6 datasets con fecha |
| Schema drift entre temporadas | Columna `season` + `.copy().reset_index()` | Dataset unificado trazable |
| CSV meteo con 3 filas de header | `skiprows=3` + `Track` inmediatamente | Ingesta correcta de 25+ ciudades |
| Ciudad ≠ Circuito | Tabla de mapeo + `.str.lower().str.strip()` + `.map()` | 24 circuitos mapeados |
| Merge silencioso por tipos de fecha | `pd.to_datetime()` explícito en ambos lados | Join efectivo |
| Duplicados por cobertura temporal | `drop_duplicates` con diagnóstico previo | Dataset sin inflación |

### Próximos pasos

1. **Datos más precisos:** reemplazar ciudades aproximadas con datos METAR/NOAA del aeropuerto más cercano al circuito.
2. **Validación temporal:** usar `TimeSeriesSplit` en lugar de `train_test_split` aleatorio para evitar data leakage entre temporadas.
3. **Clima durante la carrera:** los datos actuales son diarios. Lo ideal sería clima por hora para capturar cambios dentro de la carrera (Safety Car por lluvia, etc.).
4. **Telemetría:** integrar FIA live timing API para tener posición real de carrera, no solo la posición de llegada.